In [ ]:
# Capstone -- function 4 (4D)

import numpy as np
import matplotlib.pyplot as plt

from scipy.spatial import Delaunay, ConvexHull  # convex-hull check on the proposal

from bayes_tools import (
    load_collected, save_proposal,
    fitting,
    normalize, initial_bounds, validate_bounds_consistency,
    generate_next_point, exploit_acquisition, ucb_acquisition,
    append_observations, compute_iteration_diagnostics,
    fit_gp, get_length_scales, loo_predictions,
    compare_kappa_proposals, print_kappa_comparison,
    backtest_acquisitions, print_backtest_summary,
)
from viz_tools import (
    plot_nd_slices, plot_loo_calibration, plot_kappa_sensitivity,
    plot_acquisition_backtest,
    plot_convergence, plot_acquisition_decay, plot_uncertainty_shrinkage,
    plot_step_distance,
)

# Function 4

4D, 33 observations collected (30 initial + weeks 2–4), **32 used for modelling** — the week-2 point is out of the fit. We propose with `exploit`: pure posterior-mean maximisation, no exploration term.

Proposal this week: **`[0.402322, 0.404984, 0.373926, 0.428738]`** (GP mean +0.3514, std 0.1399, predicted improvement **+0.0828**) — a new best if it holds.

## Why the week-2 point is out of the fit

It stays in the record but not in the model. It was a **poorly chosen submission**, not a property of the function: the old notebook built bounds without a lower limit and proposed an extrapolated corner with two negative coordinates. Submitted with the signs flipped it returned **−215.0**, 6.6× worse than anything else.

It had to come out because one observation was reshaping the whole surrogate:

| | x0 | x1 | x2 | x3 | y std |
|---|---|---|---|---|---|
| excluded (live fit) | 0.860 | 0.853 | 0.890 | 0.803 | 8.00 |
| included | 1.849 | 1.986 | 1.621 | 1.690 | 34.99 |
| inflation | 2.15× | 2.33× | 1.82× | 2.11× | 4.4× |

That over-smoothing suppressed σ exactly where the search needed resolution, and it had stalled `exploit` completely — step 0.0021 from the incumbent with a predicted gain of +0.0011. With the point excluded the same acquisition steps **0.0333** and predicts **+0.0828**: 16× the step, 75× the gain.

Two side effects worth knowing. Axis coverage was 49% / 50% / **98%** / **96%** — `x2` and `x3` were stretched almost entirely by that one corner — and is now a uniform 48–49%, with the convex hull up from 11.5% to **27.0%** of the box. And `xi` means something again: against the full record's spread of 215 it was 0.005%, against the modelling spread of 32.9 it is 0.030%.

## Note: excluding it creates a risk

The surrogate no longer knows that corner is catastrophic, so nothing in the *model* stops the search going back. The **convex-hull check in the proposal cell is what stops it** — dropping the point shrinks the hull, so a corner proposal fails the test. I wouldn't weaken that check. The box check reports the corner as outside too, but only by 4e-7, which is a floating-point accident rather than protection.

## Why `exploit`

`exploit` and low-`kappa` UCB are near-identical here, and anything with real exploration weight leaves the data cloud:

| kappa | proposal | in hull | pred. mean |
|---|---|---|---|
| 0.5 | `[0.406, 0.413, 0.360, 0.431]` | yes | +0.332 |
| 1.0 | `[0.411, 0.418, 0.341, 0.432]` | yes | +0.265 |
| 1.5 | `[0.417, 0.422, 0.320, 0.433]` | yes | +0.139 |
| **2.0** | `[1.933, 1.833, 1.836, 1.917]` | **no** | **−37.49** |
| 3.0, 5.0 | same corner | **no** | −37.49 |

`exploit`'s own proposal sits 0.037 from the `kappa=1` alternative, so choosing between them is close to a formality. But the **mode switch has moved to between `kappa=1.5` and `kappa=2`**, down from between 3 and 4 before the exclusion — dropping a point moves it just as adding one does. If `exploit` is ever swapped out, `kappa <= 1.5`.

EI and PI stay unusable: every `xi` from 0.5 to 20 returned the *identical* corner. The improvement term is hopeless everywhere, so EI collapses onto its variance term and quietly becomes `max_variance`. No `xi` in any unit system fixes that.

The backtest agrees for what that's worth — `exploit` 2.056, `ucb_k1` 2.93, `ucb_k2` 7.22, down to `max_var` last at 23.16. It was always going to rank `exploit` first, so that's corroboration rather than the reason.

## What to watch

- **`exploit` has no exploration term, so it can stall.** The proposal cell warns when the step falls below 0.01. That fired last week and the exclusion cleared it; if it fires again with the outlier already out, that's a real local optimum and the fallback is `ucb` at `kappa <= 1.5`.
- **The positive region is narrow and 30 initial points missed it entirely.** The two positive observations sit 0.042 apart and 0.258 from the nearest initial point, which returned −4.03 — so `y` falls by about 4.3 over that distance. Steep, and worth remembering before trusting the GP far from the cluster.
- **All four axes are mildly relevant** — no pinned length-scales, so there's no flat subspace to restrict the search to.
- No y-scaling. `exploit` ranks by posterior mean and is exactly scale-invariant.
- LOO calibration: 32/32 points inside their own 95% interval.

In [ ]:
X_initial = np.load("initial_data/function_4/initial_inputs.npy")
y_initial = np.load("initial_data/function_4/initial_outputs.npy")
n_initial = len(y_initial)
D = X_initial.shape[1]

assert D == 4, f"Expected a 4D problem, got {D}D input -- check the loaded file."

# Loaded once and never modified.
new_X = np.empty((0, D))
new_y = np.empty((0,))

## Update this once per week

Nothing to edit here — the collected observations come from `weekly_data/function_4/`. Run `record_results.py` first so last week's result is in place.

The week-2 row carries two caveats, both in the header: the sign of `x0`/`x1` differs from what the old notebook actually proposed, and it sits about twice as far out on `x2`/`x3` as anything observed. It's excluded from the fit for that reason.

In [ ]:
# My collected observations, from weekly_data/function_4/. I run
# record_results.py before opening this notebook so the results are already
# there. pending_X holds a proposal with no result yet; it never reaches the
# model.
#
# Provenance of the collected points, oldest first:
#   - weeks 2 onward, in chronological order

new_X, new_y, pending_X = load_collected(4, D)

# The week-2 observation is excluded from the GP fit. It stays in the record --
# it's a real measurement -- but it doesn't reach the model. Full reasoning in
# the header; the short version is that one point doubles every length-scale
# (y std 35.0 vs 8.0), which stalled exploit dead, and it measures a corner we
# have since ruled out of the search.
#
# The cost: the surrogate no longer knows that corner is catastrophic, so the
# convex-hull check in the proposal cell is what keeps the search away from it.
EXCLUDE_FROM_FIT = [0]        # indices into new_X / new_y

print("observations collected so far:", len(new_y),
      f"| excluded from the fit: {EXCLUDE_FROM_FIT} -> {len(new_y) - len(EXCLUDE_FROM_FIT)} used")
print("week 2 y: %.4f (%.1fx the previous worst, %.4f)"
      % (new_y[0], new_y[0] / y_initial.min(), y_initial.min()))
print("week 3 y: %.6f | best before it: %.4f" % (new_y[-1], y_initial.max()))
print("-> %s" % ("the week-3 point is the BEST observation on record, and the"
                 " first positive y seen"
                 if new_y[-1] > y_initial.max() else "within the existing range"))
print("-> retreating to the data cloud paid off: the y spread is now %.3f,"
      " still dominated by the week-2 outlier" % (max(new_y.max(), y_initial.max())
                                                  - min(new_y.min(), y_initial.min())))


## Build the full dataset (initial + everything collected so far)

In [ ]:
# X_all / y_all: the complete record, for reporting only.
# X / y: the MODELLING set, with EXCLUDE_FROM_FIT removed. Everything downstream
# -- GP fits, acquisition, bounds, hull, diagnostics -- uses X / y.
X_all, y_all = append_observations(X_initial, y_initial, new_X, new_y)

_use = np.ones(len(new_y), bool)
_use[EXCLUDE_FROM_FIT] = False
X, y = append_observations(X_initial, y_initial, new_X[_use], new_y[_use])

print("X_initial shape:", X_initial.shape, "| y_initial shape:", y_initial.shape)
print("full record      :", X_all.shape, "| modelling set:", X.shape)
for i in EXCLUDE_FROM_FIT:
    print(f"  excluded: {np.round(new_X[i], 6)}  y = {new_y[i]:.4f}"
          f"   (see the comment in the data cell)")
print(f"y range: full record [{y_all.min():.4f}, {y_all.max():.4f}]"
      f" | modelling [{y.min():.4f}, {y.max():.4f}]")
print(f"y std:   full record {y_all.std():.3f} | modelling {y.std():.3f}")

print("\nX range per dimension (modelling set):")
print("  min:", np.round(X.min(axis=0), 4))
print("  max:", np.round(X.max(axis=0), 4))
print("y spread: %.3f | std: %.3f" % (y.max() - y.min(), y.std()))

# The domain is the unit cube, so bounds are [0,1] on every axis.
bounds = initial_bounds(X_initial, pad_fraction=1.0, lower_limit=0.0,
                        upper_limit=1.0)

# Widen only if the modelling data needs it. With the week-2 corner excluded
# nothing overshoots -- that point was the only reason this existed. Kept
# because it's harmless and would be needed again if the exclusion is undone.
overshoot = np.maximum(X.max(axis=0) - bounds[:, 1], 0.0)
if overshoot.any():
    print("\nwidening upper bounds to enclose observed data, by:", overshoot)
    bounds[:, 1] = np.maximum(bounds[:, 1], X.max(axis=0))
bounds[:, 0] = np.maximum(np.minimum(bounds[:, 0], X.min(axis=0)), 0.0)

validate_bounds_consistency(X, bounds)
print("\nBounds:\n", np.round(bounds, 4))

print("\nfraction of each bounds axis actually covered by data:")
for d in range(D):
    span = bounds[d, 1] - bounds[d, 0]
    print(f"  x{d}: {(X[:, d].max() - X[:, d].min()) / span:.1%}")

# The box is a loose stand-in for where the data actually is. At 4D with this
# few points the cloud is nearly all surface, so it's worth quantifying.
hull = Delaunay(X)
box = np.column_stack([X.min(axis=0), X.max(axis=0)])
box_volume = float(np.prod(box[:, 1] - box[:, 0]))
try:
    ch = ConvexHull(X)
    print(f"\nobserved bounding box volume: {box_volume:.4g}"
          f" | convex hull volume: {ch.volume:.4g} ({ch.volume / box_volume:.1%} of the box)")
    print(f"{len(ch.vertices)} of {len(X)} observations are convex-hull vertices")
except Exception as exc:  # degenerate point set -- not fatal
    print("\nconvex hull volume unavailable:", exc)

print(f"\nxi=0.01 would be {0.01 / (y.max() - y.min()):.4%} of the modelling y spread"
      f" ({y.max() - y.min():.3f})."
      "\nUCB needs no such correction: kappa is dimensionless.")
print("(the full record's spread is %.1f -- dominated by the excluded outlier,"
      " which is\nprecisely why any xi sized against it was meaningless)"
      % (y_all.max() - y_all.min()))

# Checking the excluded point stays out of reach -- the GP no longer knows
# it's bad.
for i in EXCLUDE_FROM_FIT:
    inside_box = bool(np.all(new_X[i] >= bounds[:, 0]) and np.all(new_X[i] <= bounds[:, 1]))
    in_hull_excl = bool(hull.find_simplex(new_X[i]) >= 0)
    print(f"\nexcluded point {np.round(new_X[i], 4)}:")
    print(f"  still inside the search box: {inside_box}"
          + ("   <- the box alone will NOT keep the search out" if inside_box else ""))
    print(f"  inside the modelling hull:   {in_hull_excl}"
          + ("   <- the hull check WILL keep the search out" if not in_hull_excl else ""))

## What the excluded point does to the fit

Recomputed each run rather than quoted, because it's the evidence for the exclusion. `gp_check` is the live surrogate on the modelling set; `gp_with` refits including the excluded point purely to show what it does to the kernel.

A single observation should not reshape a model. Here it doubles every length-scale, and the inflated length-scales and amplitude are what push variance-seeking acquisitions into the far field.

In [ ]:
# The evidence for the exclusion, recomputed rather than quoted. gp_check is the
# live surrogate; gp_with refits including the excluded point purely to show
# what it does to the kernel.
with fitting("live fit + a refit including the excluded point"):
    gp_check = fit_gp(X, y, bounds, n_restarts_optimizer=20, random_state=0)
    gp_with = fit_gp(X_all, y_all, bounds, n_restarts_optimizer=20, random_state=0)

print("LIVE fit, outlier excluded :", gp_check.kernel_)
print("refit including the outlier:", gp_with.kernel_)

ls_live = get_length_scales(gp_check)
ls_with = get_length_scales(gp_with)
print("\nlength-scales, excluded :", np.round(ls_live, 3))
print("length-scales, included :", np.round(ls_with, 3))
print("inflation from that ONE point:", np.round(ls_with / ls_live, 2))
print(f"y std: excluded {y.std():.2f} | included {y_all.std():.2f}")
print("-> a single observation more than doubling every length-scale is why it")
print("   is out of the fit. See the comment in the data cell.")

PINNED = 10.0  # fit_gp's default length_scale_bounds upper limit
pinned = [d for d, v in enumerate(ls_live) if v >= 0.999 * PINNED]
print("\npinned dimensions (GP treats as irrelevant):", pinned or "none")
print("-> all four axes are mildly relevant, so there is no flat subspace to")
print("   restrict the search to (unlike functions 2 and 3).")

## Backtest acquisition functions

Splitting the data into a seed set and a held-out candidate set, then seeing which config would have picked the best candidate. Free, and worth roughly what it costs.

`xi` here is in **raw `y`-units**, sized against the modelling spread of 32.9 rather than the default 0.01, which would be meaningless at this magnitude.

**Read it as an exploitation-phase check only.** The metric rewards whatever ranks closest to the posterior mean, so `exploit` tends to win by construction and `max_variance` to lose. `ucb_k5` also scores poorly, which happens to agree with the separate finding that high `kappa` sends the search into the far field — that agreement is a coincidence of direction, not corroboration. This metric would penalise `kappa=5` regardless.

In [ ]:
backtest_configs = [
    {"name": "ucb_k1",  "acquisition": "ucb", "kappa": 1.0},
    {"name": "ucb_k2",  "acquisition": "ucb", "kappa": 2.0},
    {"name": "ucb_k5",  "acquisition": "ucb", "kappa": 5.0},
    {"name": "ei_xi1",  "acquisition": "ei",  "xi": 1.0},
    {"name": "ei_xi5",  "acquisition": "ei",  "xi": 5.0},
    {"name": "pi_xi1",  "acquisition": "pi",  "xi": 1.0},
    {"name": "exploit", "acquisition": "exploit"},
    {"name": "max_var", "acquisition": "max_variance"},
]

with fitting("acquisition backtest, 50 splits"):
    backtest_results = backtest_acquisitions(
        X, y, bounds, backtest_configs,
        n_repeats=50, seed_frac=0.5, maximize=True,
        gp_kwargs={"n_restarts_optimizer": 15}, random_state=0,
    )

print_backtest_summary(backtest_results)

plot_acquisition_backtest(backtest_results)
plt.show()

## The kappa comparison: the record of the alternative

`exploit` takes no hyperparameter, so this cell doesn't choose anything. It's kept because it documents the mode switch that rules out high-exploration UCB, and because it keeps the conservative fallback measured rather than assumed.

What to look for is not "where do proposals stop moving" but the **jump**. On current data the proposals sit close together for `kappa` up to 1.5 — clustered near `[0.41, 0.42, 0.33, 0.43]`, inside the hull, predicted means from +0.33 down to +0.14 — and then snap to the bounds corner at `kappa >= 2` with a predicted mean of −37.49.

That jump is far-field variance winning out over a sensible posterior mean. Note how close the whole low-`kappa` band is to `exploit`'s own proposal; that closeness is what makes the choice between them nearly a formality.

Worth re-checking every week, because the switch moves. It was between `kappa=3` and `kappa=4` while the week-2 outlier was in the fit; excluding the point brought the length-scales down and moved it to between 1.5 and 2. So the usable range is now `kappa <= 1.5` if `exploit` is ever swapped out.

In [ ]:
# Both fits first, so warnings can't land in the middle of the tables.
with fitting("kappa sweep + the ucb k=1 alternative"):
    kappa_rows = compare_kappa_proposals(X, y, bounds,
                                         kappa_values=[0.5, 1.0, 1.5, 2.0, 3.0, 5.0],
                                         maximize=True, n_restarts=30, random_state=0)
    alt, _ = generate_next_point(X, y, bounds, acquisition="ucb", kappa=1.0,
                                 maximize=True, n_restarts=40, random_state=0)

print_kappa_comparison(kappa_rows)

# Flagging the mode switch explicitly rather than leaving it to be eyeballed.
print("\nper-kappa: is the proposal inside the convex hull of the data?")
for row in kappa_rows:
    inside = bool(hull.find_simplex(row["x_next"]) >= 0)
    print(f"  kappa={row['kappa']:<5g} in_hull={str(inside):<5s} pred_mean={row['pred_mean']:+10.3f}")

plot_kappa_sensitivity(X, y, bounds, gp_check, kappa_rows, ucb_acquisition, maximize=True)
plt.show()

# --- The committed choice for this function ------------------------------
# exploit = pure posterior-mean maximisation; takes no hyperparameter.
# Conservative fallback, if convergence stalls or the surface looks multi-modal:
#     ACQ, ACQ_KWARGS = "ucb", {"kappa": 1.0}
ACQ = "exploit"
ACQ_KWARGS = {}

print(f"\nusing acquisition = {ACQ!r} {ACQ_KWARGS}")
print("  no hyperparameter, and exactly scale-invariant (ranks by posterior mean),")
print("  so no y-scaling is required.")

# How far is exploit's choice from the low-kappa UCB alternative? (fitted above)
print(f"\nucb kappa=1 would propose: {np.round(alt, 6)}")

## Propose the next point

Two checks on the result. The **per-axis** check asks whether each coordinate is within the range already observed on its own axis. The **convex-hull** check is strictly stronger and is the one that matters here — a point can pass the per-axis test and still sit outside the data cloud entirely, which is exactly what the `kappa >= 2` proposals do.

Bounds here are the project-standard ones rather than an observed box, and no axis restriction is needed: with the outlier excluded, `exploit` and low-`kappa` UCB both land inside the hull on their own.

In [ ]:
with fitting("the committed proposal"):
    x_next, gp = generate_next_point(
        X, y, bounds,
        acquisition=ACQ,  # see the comparison above; "exploit" takes no hyperparameter
        maximize=True,
        n_restarts=40,
        random_state=0,
        **ACQ_KWARGS,
    )

print(f"\n--- Next point to evaluate (bounds shape {bounds.shape}) ---")
print("x_next:", np.round(x_next, 6))
print(gp.kernel_)

mu, sigma = gp.predict(normalize(x_next.reshape(1, -1), bounds), return_std=True)
print(f"GP predicted mean: {mu[0]:.4g}, predicted std: {sigma[0]:.4g}")
print(f"current best observed y: {y.max():.4g}"
      f"  -> predicted improvement: {mu[0] - y.max():+.4g}")

print(f"\ndistance from the ucb kappa=1 alternative: {np.linalg.norm(x_next - alt):.4f}")

# --- Safety checks -------------------------------------------------------
on_edge = [d for d in range(D)
           if np.isclose(x_next[d], bounds[d, 0]) or np.isclose(x_next[d], bounds[d, 1])]
outside = [d for d in range(D) if not (X[:, d].min() <= x_next[d] <= X[:, d].max())]
print("\ndimensions where x_next sits on a bound:", on_edge or "none")
print("dimensions where x_next is outside the observed data range:", outside or "none")

in_hull = bool(hull.find_simplex(x_next) >= 0)
print(f"x_next inside the CONVEX HULL of the observations: {in_hull}")
if not in_hull:
    print("*** x_next is outside the data cloud -- that is extrapolation, and this")
    print("    function has already shown what extrapolation costs (y = -215). ***")

# exploit has no exploration term, so watch for it stalling on the incumbent.
d_inc = np.linalg.norm(x_next - X[np.argmax(y)])
print(f"\ndistance from the incumbent: {d_inc:.4f}")
if d_inc < 0.01:
    print("*** exploit is proposing essentially the incumbent again -- it has")
    print("    stopped making progress. Switch to ucb with a small kappa. ***")

print("nearest 3 observations to x_next:")
for i in np.argsort(np.linalg.norm(X - x_next, axis=1))[:3]:
    print(f"  dist={np.linalg.norm(X[i] - x_next):.4f}  y={y[i]:+9.3f}  X={np.round(X[i], 3)}")

## Visualise the GP and acquisition function via 1D slices

Each panel holds the other three dimensions at the best observed point and sweeps one. Dotted line is the fixed centre, dashed red is `x_next`.

Because the acquisition is `exploit`, the green curve is the posterior mean itself and tracks the blue GP mean exactly. That's expected, and it makes the reason for the choice visible: the mean has a clear interior optimum on each axis while the *uncertainty* band widens toward the far ends. Any acquisition weighting that band heavily follows it outward, which is what the rejected configs do.

A partial view — it shows what the GP believes along each axis near the best point, not interactions.

In [ ]:
# exploit_acquisition is just the posterior mean, so the green acquisition curve
# below traces the GP mean itself -- that is expected, not a plotting bug.
plot_nd_slices(
    X, y, bounds, gp,
    acquisition_fn=exploit_acquisition,
    x_next=x_next,
    acq_kwargs={"maximize": True},
)
plt.show()

## Sanity-check the surrogate: leave-one-out calibration

Refitting once per observation, holding it out and predicting it from the rest. At D=4 this is the main way to judge the surrogate, since the fitted surface can't be inspected directly.

Read the error bars rather than the correlation. Note the excluded outlier isn't in this set by construction, so what's measured here is the surrogate actually in use rather than the one it distorted.

In [ ]:
with fitting("leave-one-out calibration"):
    pred_mean, pred_std = loo_predictions(X, y, bounds,
                                          gp_kwargs={"n_restarts_optimizer": 20})

plot_loo_calibration(y, pred_mean, pred_std)
plt.show()

within = np.abs(y - pred_mean) <= 1.96 * pred_std
print(f"points inside their own 95% LOO interval: {within.sum()}/{len(y)}")

# NOTE: this LOO runs on the MODELLING set, so the week-2 outlier is not in it.
# The old version asked "is the worst-predicted point the collected outlier?" by
# testing `worst == n_initial`; with the outlier excluded that index is week 3,
# so the question no longer means anything. Report which point it is instead.
worst = int(np.argmax(np.abs(y - pred_mean)))
src = "initial batch" if worst < n_initial else f"collected #{worst - n_initial + 1}"
print(f"worst-predicted point: index {worst} ({src}) -> true {y[worst]:.4g},"
      f" predicted {pred_mean[worst]:.4g} (std {pred_std[worst]:.4g})")
print("(the excluded week-2 outlier is absent by construction, so this measures")
print(" the surrogate actually in use rather than the one distorted by it)")

## Iteration diagnostics

Replaying the ordered observations to reconstruct the acquisition value, GP hyperparameters and domain-wide uncertainty at each past proposal.

It applies one setting to the whole history, taken from `ACQ`/`ACQ_KWARGS` so it can't drift from what the proposal used. The history is mixed — week 2 came from the old notebook's settings — so that row's replayed acquisition value describes a decision that was never made that way.

One convenient property of replaying as `exploit`: the acquisition value is just the posterior mean at the point *before* it was evaluated, so it reads as "what did the model expect there", which is interpretable however the point was chosen. Comparing it against the `y` actually observed shows how badly the model was surprised.

The replay counts iterations from the modelling set, not the full record, so the excluded row doesn't shift the numbering. Row order has to be true chronological order. `plot_bo_diagnostics` is skipped — it hard-codes a 2D scatter panel — and the trend plots below are gated on having a few completed iterations.

In [ ]:
# The old `if len(new_y) == 0` guard is gone: new_y is populated above, so that
# branch was unreachable.
#
# The count must come from the MODELLING set, not len(new_y). The replay only
# sees the rows actually fitted, so with the week-2 outlier excluded there are
# len(y) - n_initial iterations, not len(new_y). The old code used len(new_y)
# and would have over-reported by one and mis-gated the trend plots.
n_replayed = len(y) - n_initial

with fitting("iteration-diagnostics replay"):
    history = compute_iteration_diagnostics(X, y, bounds, n_initial=n_initial,
                                            acquisition=ACQ, maximize=True, **ACQ_KWARGS)

print("Best y so far:", np.nanmax(history["y"]))
print(f"completed iterations in the modelling set: {n_replayed}"
      f"  ({len(new_y)} collected, {len(new_y) - n_replayed} excluded from the fit)")

if n_replayed >= 3:
    for plot_fn in (plot_convergence, plot_acquisition_decay,
                    plot_uncertainty_shrinkage, plot_step_distance):
        plot_fn(history)
        plt.show()
else:
    print(f"\nOnly {n_replayed} replayed iteration(s) -- need at least 3 before the")
    print("trend plots say anything. Skipping them; the raw fields are below.")

## Raw diagnostic fields

In [ ]:
print("y:", np.round(history["y"], 4))
print("\niteration:", history["iteration"])
print("\nacq_value (NaN = initial batch; see the caveat above):", history["acq_value"])
print("\nlength_scale:", history["length_scale"])
print("\ndomain_mean_std:", history["domain_mean_std"])

In [ ]:
# My proposal as a hyphen-separated string, for submission.
print("-".join(f"{v:.6f}" for v in x_next))

# Full precision as well. 6 dp is fine to submit, but paste THIS into next
# week's new_X: a 6-dp copy of THIS function's previous proposal rounded 4e-7
# outside its own upper bound and tripped validate_bounds_consistency.
print("\nfull precision (use for next week's new_X):")
print("-".join(repr(float(v)) for v in x_next))

In [ ]:
# I record this week's proposal in the weekly store. This rewrites inputs.csv
# rather than appending, so re-running the notebook cannot double-record. I
# submit the 6-dp string printed above -- it is the same value written here.
save_proposal(4, new_X, x_next)
